# Titanic ④ 欠損値を補う

欠損のある列（Age，Cabin，Embarked，Fare）をどう扱うかを，データを見ながら決めた記録．

比べるときは，5 分割交差検証を分け方を変えながら 5 回繰り返し（計 25 回），その平均で判断する．1 回だけの 5 分割だと，分け方の運で平均が 0.01 程度動いてしまい，埋め方の違いによる差より大きくなるため．比べるモデルは 02 と同じ設定のロジスティック回帰とランダムフォレスト．

ここで決めた特徴量と埋め方は `src/titanic/features.py` に定数としてまとめてあり，05 以降はそれを使う．

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from titanic.data import load_data
from titanic.features import CATEGORICAL, IMPUTE_BY, NUMERIC, fill_embarked
from titanic.modeling import build_pipeline, cv_accuracy, summarize

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

train, test = load_data()
y = train["Survived"]

MODELS = {
    "ロジスティック回帰": LogisticRegression(max_iter=1000),
    "ランダムフォレスト": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=0),
}


def compare(settings: dict[str, dict], X: pd.DataFrame) -> pd.DataFrame:
    """設定ごとに，2 つのモデルの交差検証の正解率（平均と標準偏差）を並べた表を返す．

    settings は，設定の名前から build_pipeline() に渡す引数（numeric，categorical，impute_by）への辞書．
    """
    return pd.concat(
        {
            model_name: summarize({name: cv_accuracy(build_pipeline(model, **kwargs), X, y) for name, kwargs in settings.items()})
            for model_name, model in MODELS.items()
        },
        axis=1,
    )

## 欠損の状況

In [ ]:
missing = pd.concat({"train": train.isna().sum(), "test": test.isna().sum()}, axis=1).astype("Int64")
print(missing[missing.sum(axis=1) > 0])

## 欠損のある行を捨てない理由

いちばん簡単なのは欠損のある行を捨てることだが，test の行は捨てられない（418 人全員の予測を提出する必要がある）．train の行だけ捨てる手もあるが，学習に使える人数が減るうえ，捨てる人に偏りがある．

In [ ]:
age_missing = train["Age"].isna()
print(f"Age が欠けている行を捨てると: {len(train)} 行 → {(~age_missing).sum()} 行")
print(f"生存率  Age が欠けている人: {y[age_missing].mean():.3f}  欠けていない人: {y[~age_missing].mean():.3f}")
print("等級ごとの Age の欠損率:", train.groupby("Pclass")["Age"].apply(lambda s: s.isna().mean()).round(2).to_dict())

Age が欠けている人は生存率が低く，3 等に多い．捨てると，3 等で亡くなった人が減った，偏ったデータで学習することになる．

## Age

年齢は名前の敬称によって大きく違う．全体の中央値で埋めると，年齢が分からない男の子（Master）も大人と同じ年齢になってしまう．

In [ ]:
# 名前（"Braund, Mr. Owen Harris" の形）から敬称を取り出し，人数の少ない敬称は Rare にまとめる
title = train["Name"].str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
title = title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
title = title.where(title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
X_age = fill_embarked(train).assign(Title=title, AgeMissing=age_missing.astype(int))

print(f"全体の中央値: {train['Age'].median()}")
print(X_age.groupby("Title")["Age"].agg(人数="size", 欠損=lambda s: s.isna().sum(), 中央値="median"))

埋め方を 4 通りに変えて比べる．「欠損フラグ」は，Age が欠けていれば 1 になる列 AgeMissing を特徴量に足すこと．欠けていたという情報そのものをモデルに渡せる．

In [ ]:
with_flag = NUMERIC + ["AgeMissing"]
by_title = {"Age": "Title", **IMPUTE_BY}
age_settings = {
    "全体の中央値": dict(numeric=NUMERIC, categorical=CATEGORICAL, impute_by=IMPUTE_BY),
    "全体の中央値＋欠損フラグ": dict(numeric=with_flag, categorical=CATEGORICAL, impute_by=IMPUTE_BY),
    "敬称ごとの中央値": dict(numeric=NUMERIC, categorical=CATEGORICAL, impute_by=by_title),
    "敬称ごとの中央値＋欠損フラグ": dict(numeric=with_flag, categorical=CATEGORICAL, impute_by=by_title),
}
print(compare(age_settings, X_age))

4 通りの差は，どちらのモデルでも最大 0.003 だった．標準偏差（0.026〜0.028）よりずっと小さく，このデータでは埋め方による差は見えなかった．

**決定: Age は全体の中央値で埋める（02 と同じ）．**

## Cabin

In [ ]:
has_cabin = train["Cabin"].notna()
print(f"欠損率: {1 - has_cabin.mean():.2f}")
print(f"生存率  記録あり: {y[has_cabin].mean():.3f}  記録なし: {y[~has_cabin].mean():.3f}")
print("等級ごとの記録率:", train.groupby("Pclass")["Cabin"].apply(lambda s: s.notna().mean()).round(2).to_dict())

8 割近くが欠けているので，値を推測して埋めるのは現実的ではない．記録の有無で生存率は大きく違うが，記録があるのはほとんどが 1 等の人で，等級とほぼ同じ情報になっている．

**決定: Cabin は使わない．**

## Embarked と Fare

In [ ]:
print(train.loc[train["Embarked"].isna(), ["PassengerId", "Pclass", "Sex", "Fare", "Ticket", "Cabin"]])
print()
print("1 等の乗船港ごとの運賃:")
print(train[train["Pclass"] == 1].groupby("Embarked")["Fare"].agg(中央値="median", 人数="size"))
print()
print(test.loc[test["Fare"].isna(), ["PassengerId", "Pclass", "Sex", "Age", "Embarked"]])
print("等級ごとの運賃の中央値:", train.groupby("Pclass")["Fare"].median().round(2).to_dict())

Embarked が欠けている 2 人は，同じチケットの 1 等の乗客で，運賃は 80．1 等の運賃の中央値は C が 78.3，S が 52.0 で，C に近い．Fare が欠けているのは test にいる 3 等の 1 人．

**決定: Embarked は C で，Fare は 3 等の中央値で埋める．**

Fare の中央値は学習データから計算する値なので，Pipeline の中で埋める（`IMPUTE_BY`）．どちらも件数がわずかで，交差検証のスコアにはほとんど影響しない．Fare の欠損は test にしかないので，交差検証にはそもそも出てこない．

## 特徴量を足すかどうか

03 で候補に挙げた敬称（Title）と家族の人数の区分（FamilyGroup: 1 人，2〜4 人，5 人以上）を足した場合と比べる．Age はここで決めたとおり全体の中央値で埋める．

In [ ]:
family_size = train["SibSp"] + train["Parch"] + 1
X_feat = X_age.assign(
    FamilyGroup=pd.cut(family_size, bins=[0, 1, 4, 11], labels=["alone", "small", "large"]).astype(str)
)
feature_settings = {
    "A: 02 と同じ 7 列": dict(numeric=NUMERIC, categorical=CATEGORICAL, impute_by=IMPUTE_BY),
    "B: A＋敬称": dict(numeric=NUMERIC, categorical=CATEGORICAL + ["Title"], impute_by=IMPUTE_BY),
    "C: A＋家族の人数の区分": dict(numeric=NUMERIC, categorical=CATEGORICAL + ["FamilyGroup"], impute_by=IMPUTE_BY),
    "D: A＋敬称＋家族の人数の区分": dict(numeric=NUMERIC, categorical=CATEGORICAL + ["Title", "FamilyGroup"], impute_by=IMPUTE_BY),
}
print(compare(feature_settings, X_feat))

敬称を足すと，ロジスティック回帰は 0.800 から 0.827 に上がる．ランダムフォレストの上がり幅は 0.008 と小さい．家族の人数の区分は，ロジスティック回帰で 0.009 上がる程度だった．

**決定: 特徴量は 02 と同じ 7 列のまま進める．**

## 決定のまとめ

| 対象 | 決めたこと |
|---|---|
| Age | 全体の中央値で埋める |
| Cabin | 使わない |
| Embarked | C で埋める |
| Fare（test の 1 人） | 3 等の中央値で埋める |
| 特徴量 | 02 と同じ 7 列（Pclass，Sex，Age，SibSp，Parch，Fare，Embarked） |

この設定での交差検証の正解率は，ロジスティック回帰が 0.800，ランダムフォレストが 0.821（上の表の A）．05 ではこれを基準にモデルを比べる．